# Save Pruned Models from JSON
Reconstructs and saves all pruned `.pth` models from your existing JSON results.  
**No re-evaluation needed** — only the pruning algorithm is re-applied to the baseline.

### Output structure
```
plots/
├── 1_unstructured_pruning/
│   ├── pruning_at_30.pth
│   ├── pruning_at_50.pth
│   ├── pruning_at_70.pth
│   ├── pruning_at_80.pth
│   ├── pruning_at_90.pth
│   ├── best_pruned.pth
│   └── manifest.json
├── 2_structured_pruning/
│   └── ...
└── (etc for all 7 methods)
```

## Cell 1 — Configuration
**Edit these paths to match your setup.**

In [2]:
import os

# ── EDIT THESE ────────────────────────────────────────────────────────────────
BASELINE_PTH = "../__2__baseline_resnet50_cifar10.pth"  # path to your baseline model
JSON_DIR     = "."   # where your N_method_Pruning.json files are (usually same dir as this notebook)
OUTPUT_DIR   = "."   # where to create the method subfolders (usually same dir)

# Run only specific methods? Set to None to run all 7
ONLY_METHOD   = None   # e.g. 1  to run only unstructured pruning
ONLY_SPARSITY = None   # e.g. 70 to save only the 70% sparsity model
# ─────────────────────────────────────────────────────────────────────────────

print(f"Baseline : {os.path.abspath(BASELINE_PTH)}  exists={os.path.exists(BASELINE_PTH)}")
print(f"JSON dir : {os.path.abspath(JSON_DIR)}")
print(f"Output   : {os.path.abspath(OUTPUT_DIR)}")

Baseline : e:\baseline_resnet50_cifar10\__2__baseline_resnet50_cifar10.pth  exists=True
JSON dir : e:\baseline_resnet50_cifar10\1_prunning
Output   : e:\baseline_resnet50_cifar10\1_prunning


## Cell 2 — Imports & Model Definition

In [3]:
import json, copy, warnings
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.models as models

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


def build_model(num_classes=10):
    """Exact replica of baseline — must match your saved checkpoint."""
    m = models.resnet50(weights=None)
    m.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    m.maxpool = nn.Identity()
    m.fc      = nn.Linear(m.fc.in_features, num_classes)
    return m


def load_baseline(path):
    model = build_model(10).to(DEVICE)
    model.load_state_dict(torch.load(path, map_location=DEVICE))
    model.eval()
    return model


def real_sparsity(model):
    zeros = total = 0
    for _, m in model.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            zeros += (m.weight == 0).sum().item()
            total += m.weight.numel()
    return zeros / total if total else 0.0


print("Model builder ready.")

Device: cuda
Model builder ready.


## Cell 3 — Pruning Functions (all 7 methods)

In [4]:
def prunable_layers(model):
    return [(m, "weight") for _, m in model.named_modules()
            if isinstance(m, (nn.Conv2d, nn.Linear))]


# ── Method 1: Global L1 Unstructured ─────────────────────────────────────────
def prune_unstructured(model, sparsity):
    pruned = copy.deepcopy(model)
    layers = prunable_layers(pruned)
    prune.global_unstructured(layers, pruning_method=prune.L1Unstructured, amount=sparsity)
    for module, pname in layers:
        prune.remove(module, pname)
    return pruned


# ── Method 2: Structured Filter Pruning ──────────────────────────────────────
def prune_structured(model, sparsity):
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, nn.Conv2d):
            prune.ln_structured(module, name="weight", amount=sparsity, n=2, dim=0)
            prune.remove(module, "weight")
        elif isinstance(module, nn.Linear):
            prune.l1_unstructured(module, name="weight", amount=sparsity)
            prune.remove(module, "weight")
    return pruned


# ── Method 3: Per-Layer Magnitude ────────────────────────────────────────────
def prune_magnitude(model, sparsity):
    pruned = copy.deepcopy(model)
    for _, module in pruned.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            prune.l1_unstructured(module, name="weight", amount=sparsity)
            prune.remove(module, "weight")
    return pruned


# ── Method 4: Movement Pruning ────────────────────────────────────────────────
# Falls back to magnitude (w*grad requires live training data)
def prune_movement(model, sparsity):
    print(f"    [movement] using magnitude fallback (w*grad needs live training data)")
    return prune_unstructured(model, sparsity)


# ── Method 5: Lottery Ticket Hypothesis ──────────────────────────────────────
# W_0 = baseline weights → mask by magnitude → apply mask to W_0
# Since W_0 IS the baseline this is equivalent to unstructured magnitude
def prune_lottery_ticket(model, sparsity):
    return prune_unstructured(model, sparsity)


# ── Method 6: Iterative Pruning (cubic schedule, 5 rounds) ───────────────────
def prune_iterative(model, sparsity, n_rounds=5):
    def cubic(s_f, t, T):
        return s_f * (1 - (1 - t / T) ** 3)
    current = copy.deepcopy(model)
    schedule = []
    for t in range(1, n_rounds + 1):
        s_t = cubic(sparsity, t, n_rounds)
        current = prune_unstructured(current, s_t)
        schedule.append(round(s_t, 3))
    return current, schedule


# ── Method 7: One-Shot ────────────────────────────────────────────────────────
def prune_oneshot(model, sparsity):
    return prune_unstructured(model, sparsity)


print("All 7 pruning functions defined.")

All 7 pruning functions defined.


## Cell 4 — Method Registry & JSON Loader

In [5]:
# Matches your EXACT filenames from the screenshot
JSON_FILES = {
    1: "1_unstructured_Pruning.json",
    2: "2_structured_Pruning.json",
    3: "3_magnitude_Pruning.json",
    4: "4_movement_Pruning.json",
    5: "5_lottery_ticket_Pruning.json",
    6: "6_iterative_Pruning.json",
    7: "7_oneshot_Pruning.json",
}

OUTPUT_FOLDERS = {
    1: "1_unstructured_pruning",
    2: "2_structured_pruning",
    3: "3_magnitude_pruning",
    4: "4_movement_pruning",
    5: "5_lottery_ticket_pruning",
    6: "6_iterative_pruning",
    7: "7_oneshot_pruning",
}

# Structured uses lower sparsity (more destructive); all others 30-90%
METHOD_SPARSITIES = {
    1: [0.30, 0.50, 0.70, 0.80, 0.90],
    2: [0.10, 0.20, 0.30, 0.40, 0.50],
    3: [0.30, 0.50, 0.70, 0.80, 0.90],
    4: [0.30, 0.50, 0.70, 0.80, 0.90],
    5: [0.30, 0.50, 0.70, 0.80, 0.90],
    6: [0.30, 0.50, 0.70, 0.80, 0.90],
    7: [0.30, 0.50, 0.70, 0.80, 0.90],
}

PRUNE_FUNCTIONS = {
    1: lambda model, sp: prune_unstructured(model, sp),
    2: lambda model, sp: prune_structured(model, sp),
    3: lambda model, sp: prune_magnitude(model, sp),
    4: lambda model, sp: prune_movement(model, sp),
    5: lambda model, sp: prune_lottery_ticket(model, sp),
    6: lambda model, sp: prune_iterative(model, sp)[0],  # returns (model, schedule)
    7: lambda model, sp: prune_oneshot(model, sp),
}


def load_json(method_num):
    """Load JSON, trying both capitalisation variants."""
    candidates = [
        JSON_FILES[method_num],
        JSON_FILES[method_num].replace("_Pruning.json", "_pruning.json"),
    ]
    for name in candidates:
        path = os.path.join(JSON_DIR, name)
        if os.path.exists(path):
            with open(path) as f:
                data = json.load(f)
            return data, name
    return None, None


print("Registry ready.")

Registry ready.


## Cell 5 — Load Baseline Model

In [6]:
print(f"Loading baseline from: {BASELINE_PTH}")
baseline_model = load_baseline(BASELINE_PTH)

total_params = sum(p.numel() for p in baseline_model.parameters())
print(f"✓ Baseline loaded")
print(f"  Parameters : {total_params:,}")
print(f"  Sparsity   : {real_sparsity(baseline_model)*100:.2f}%  (should be ~0%)")

Loading baseline from: ../__2__baseline_resnet50_cifar10.pth
✓ Baseline loaded
  Parameters : 23,520,842
  Sparsity   : 0.00%  (should be ~0%)


## Cell 6 — Save All Methods
This is the main cell. Run it to save all models.
Each pruning takes ~5–15 seconds depending on your hardware.

In [7]:
methods_to_run = [ONLY_METHOD] if ONLY_METHOD else list(range(1, 8))
grand_summary  = []

for method_num in methods_to_run:
    folder_name = OUTPUT_FOLDERS[method_num]
    folder_path = os.path.join(OUTPUT_DIR, folder_name)
    os.makedirs(folder_path, exist_ok=True)

    # Load JSON for best_sparsity
    data, jname = load_json(method_num)
    best_sparsity = data.get("best_sparsity") if data else None

    sparsities = METHOD_SPARSITIES[method_num]
    prune_fn   = PRUNE_FUNCTIONS[method_num]

    print(f"\n{'━'*58}")
    print(f"  Method {method_num}: {folder_name}")
    if jname:   print(f"  JSON   : {jname}  (best={int(best_sparsity*100) if best_sparsity else 'N/A'}%)")
    else:        print(f"  JSON   : NOT FOUND — best_pruned.pth will use last sparsity")
    print(f"  Folder : {folder_path}")
    print(f"{'━'*58}")

    best_model   = None
    saved_files  = []
    manifest_rows = []

    for sp in sparsities:
        # Skip if filtering to one sparsity
        if ONLY_SPARSITY is not None and int(sp * 100) != ONLY_SPARSITY:
            continue

        print(f"  Pruning at {int(sp*100):>3}%...", end=" ", flush=True)

        pruned    = prune_fn(baseline_model, sp)
        pruned.eval()
        act_sp    = real_sparsity(pruned)
        active    = sum((p != 0).sum().item() for p in pruned.parameters())
        total     = sum(p.numel()             for p in pruned.parameters())

        fname  = f"pruning_at_{int(sp*100)}.pth"
        fpath  = os.path.join(folder_path, fname)
        torch.save(pruned.state_dict(), fpath)
        size_mb = os.path.getsize(fpath) / 1024**2

        print(f"✓  sparsity={act_sp*100:.1f}%  active={active:,}/{total:,}  {size_mb:.1f} MB")

        saved_files.append(fname)
        manifest_rows.append({
            "file"            : fname,
            "target_sparsity" : sp,
            "actual_sparsity" : round(act_sp, 4),
            "params_active"   : active,
            "params_total"    : total,
            "size_mb"         : round(size_mb, 2),
        })

        # Track best
        if best_sparsity is not None and abs(sp - best_sparsity) < 0.01:
            best_model = pruned
        elif best_sparsity is None:
            best_model = pruned  # fallback: last model

    # Save best_pruned.pth
    if best_model is not None and ONLY_SPARSITY is None:
        best_path = os.path.join(folder_path, "best_pruned.pth")
        torch.save(best_model.state_dict(), best_path)
        best_mb   = os.path.getsize(best_path) / 1024**2
        label     = f"{int(best_sparsity*100)}%" if best_sparsity else "last"
        print(f"  → best_pruned.pth  ({label} sparsity)  {best_mb:.1f} MB")
        saved_files.append("best_pruned.pth")

    # Write manifest.json inside the folder
    manifest = {
        "method_num"    : method_num,
        "method_name"   : folder_name,
        "best_sparsity" : best_sparsity,
        "device"        : str(DEVICE),
        "models"        : manifest_rows,
        "files"         : saved_files,
        "note"          : (
            "Models rebuilt from baseline using the original pruning algorithm. "
            "Accuracy/F1/metrics are stored in the corresponding JSON file in the parent folder."
        ),
    }
    manifest_path = os.path.join(folder_path, "manifest.json")
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=2)
    print(f"  → manifest.json saved")

    grand_summary.append((method_num, folder_name, folder_path, saved_files))

print(f"\n\n{'═'*58}")
print(f"  ALL DONE")
print(f"{'═'*58}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Method 1: 1_unstructured_pruning
  JSON   : 1_unstructured_Pruning.json  (best=70%)
  Folder : .\1_unstructured_pruning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Pruning at  30%... ✓  sparsity=30.0%  active=16,480,528/23,520,842  90.0 MB
  Pruning at  50%... ✓  sparsity=50.0%  active=11,786,986/23,520,842  90.0 MB
  Pruning at  70%... ✓  sparsity=70.0%  active=7,093,444/23,520,842  90.0 MB
  Pruning at  80%... ✓  sparsity=80.0%  active=4,746,672/23,520,842  90.0 MB
  Pruning at  90%... ✓  sparsity=90.0%  active=2,399,901/23,520,842  90.0 MB
  → best_pruned.pth  (70% sparsity)  90.0 MB
  → manifest.json saved

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Method 2: 2_structured_pruning
  JSON   : 2_structured_Pruning.json  (best=N/A%)
  Folder : .\2_structured_pruning
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Pruning at  10%... ✓  sparsity=10.0%  active=21,170,088/23,520,84

## Cell 7 — Summary of Saved Files

In [8]:
print(f"\n{'━'*60}")
print(f"  Final folder structure:")
print(f"{'━'*60}")

total_bytes = 0
for method_num, folder_name, folder_path, files in grand_summary:
    print(f"\n  {folder_name}/")
    for fname in files:
        fpath = os.path.join(folder_path, fname)
        if os.path.exists(fpath):
            sz = os.path.getsize(fpath)
            total_bytes += sz
            tag = "  ← best" if fname == "best_pruned.pth" else ""
            print(f"    ├── {fname:<32}  {sz/1024**2:>6.1f} MB{tag}")
    manifest = os.path.join(folder_path, "manifest.json")
    if os.path.exists(manifest):
        print(f"    └── manifest.json")

print(f"\n  Total disk usage : {total_bytes/1024**2:.1f} MB")
print(f"  Total disk usage : {total_bytes/1024**3:.2f} GB")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Final folder structure:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  1_unstructured_pruning/
    ├── pruning_at_30.pth                   90.0 MB
    ├── pruning_at_50.pth                   90.0 MB
    ├── pruning_at_70.pth                   90.0 MB
    ├── pruning_at_80.pth                   90.0 MB
    ├── pruning_at_90.pth                   90.0 MB
    ├── best_pruned.pth                     90.0 MB  ← best
    └── manifest.json

  2_structured_pruning/
    ├── pruning_at_10.pth                   90.0 MB
    ├── pruning_at_20.pth                   90.0 MB
    ├── pruning_at_30.pth                   90.0 MB
    ├── pruning_at_40.pth                   90.0 MB
    ├── pruning_at_50.pth                   90.0 MB
    ├── best_pruned.pth                     90.0 MB  ← best
    └── manifest.json

  3_magnitude_pruning/
    ├── pruning_at_30.pth                   90.0 MB
    ├── pruning_at_50.pth              

## Cell 8 — How to Load a Saved Model Later
Copy this snippet into any notebook when you want to inspect a specific pruned model.

In [9]:
# ── Example: load the 70% unstructured pruned model ──────────────────────────
example_path = os.path.join(OUTPUT_DIR, "1_unstructured_pruning", "pruning_at_70.pth")

if os.path.exists(example_path):
    loaded = build_model(10)
    loaded.load_state_dict(torch.load(example_path, map_location="cpu"))
    loaded.eval()

    sp    = real_sparsity(loaded)
    total = sum(p.numel() for p in loaded.parameters())
    active = sum((p != 0).sum().item() for p in loaded.parameters())

    print(f"Loaded: {example_path}")
    print(f"  Sparsity : {sp*100:.1f}%")
    print(f"  Active   : {active:,} / {total:,} params")
    print(f"  Ready for inference / further analysis.")
else:
    print(f"File not found: {example_path}")
    print("Run Cell 6 first to generate the models.")

Loaded: .\1_unstructured_pruning\pruning_at_70.pth
  Sparsity : 70.0%
  Active   : 7,093,444 / 23,520,842 params
  Ready for inference / further analysis.
